# 02 — Model comparison
       Compare 4 models on the validation set (last ~2 months of train).
       Test set is NOT used here — it's reserved for the final model only.


## Setup

In [1]:
import os
import sys
import time
from pathlib import Path

import pandas as pd

# Run from project root so relative paths (data/raw) and `app` imports work.
# The check makes it safe to re-run this cell (otherwise each run goes one level up).
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

from app.data import load_data, split_xy
from app.features import build_features
from app.metrics import evaluate
from app.models import MODELS

## Data
Time-based split: train (fit) / val (compare) / test (untouched).

In [2]:
# load_data() concatenates train + test, so history features for test see train's past
splits = split_xy(build_features(load_data()))

X_train, y_train = splits["train"]
X_val, y_val = splits["val"]

print(f"train: {X_train.shape}, fraud rate {y_train.mean():.2%}")
print(f"val:   {X_val.shape}, fraud rate {y_val.mean():.2%}")

train: (1142581, 10), fraud rate 0.57%
val:   (154094, 10), fraud rate 0.62%


## Train and evaluate

Each model is fitted on train and scored on val. Main metric: PR-AUC
(ROC-AUC is shown for reference — it looks optimistic under heavy imbalance).

In [3]:
results = []

for name, make in MODELS.items():
    model = make()  # fresh untrained model

    start = time.time()
    model.fit(X_train, y_train)
    fit_seconds = time.time() - start

    # Score both sets: a big train/val gap = the model memorised train (overfitting)
    train_scores = model.predict_proba(X_train)[:, 1]
    val_scores = model.predict_proba(X_val)[:, 1]

    train_pr = evaluate(y_train, train_scores)["pr_auc"]
    val_metrics = evaluate(y_val, val_scores)

    results.append({
        "model": name,
        "train_pr_auc": train_pr,
        **val_metrics,                                   # val pr_auc + roc_auc
        "gap": round(train_pr - val_metrics["pr_auc"], 4),
        "fit_seconds": round(fit_seconds, 1),
    })
    print(name, f"train {train_pr} | val {val_metrics['pr_auc']}")

logreg train 0.3568 | val 0.3615
tree train 0.7117 | val 0.6917
xgb train 0.8174 | val 0.7965
lgbm train 0.848 | val 0.8019


## Results

In [4]:
results_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False)
results_df

,model,train_pr_auc,pr_auc,roc_auc,gap,fit_seconds
3,lgbm,0.8480,0.8019,0.9934,0.0461,5.5
2,xgb,0.8174,0.7965,0.9926,0.0209,7.1
1,tree,0.7117,0.6917,0.9694,0.0200,5.5
0,logreg,0.3568,0.3615,0.9135,-0.0047,1.3


**Conclusion:** With default parameters XGBoost led clearly (val PR-AUC 0.795 vs 0.362
for the logistic regression baseline, 2.2x). A single decision tree already doubles the
baseline, which points to strong non-linear interactions (e.g. night + online + high amount).

LightGBM with defaults underperformed even a single tree: it was weaker on train as well
(0.74 vs 0.83) and had the largest train/val gap — unstable training under heavy class
imbalance, not just overfitting. With regularisation (lower learning rate, more trees,
smaller leaves) it recovered to 0.802.

For a fair comparison XGBoost got the same treatment: 0.797, gap 0.021. The two boosted
models are within 0.01 of each other, so the decision rule set in advance applies —
pick the smaller train/val gap. **Final model: XGBoost (tuned)**, which is also robust
to hyperparameters (0.795 → 0.797), unlike LightGBM (0.64 → 0.80).

ROC-AUC barely separates the models (0.91–0.99), confirming PR-AUC as the primary metric.